In [0]:
from src.claims_project.config import TABLES

In [0]:
tables_to_validate = [
    TABLES["bronze_claims"],
    TABLES["silver_claims"],
    TABLES["silver_providers"],
    TABLES["silver_claims_enriched"],
    TABLES["gold_claim_lag"],
    TABLES["gold_provider_monthly"],
    TABLES["gold_suspicious_claims"]
]

In [0]:
for table_name in tables_to_validate:
    print(f"\nValidating table: {table_name}")
    df = spark.table(table_name)
    row_count = df.count()
    print(f"Row count: {row_count}")
    assert row_count > 0, f"{table_name} is empty"

In [0]:
null_claims = spark.sql("""
SELECT COUNT(*) AS cnt
FROM dev.claims_project.silver_claims
WHERE ClaimID IS NULL
""").collect()[0]["cnt"]

assert null_claims == 0, "Null ClaimID values found"


In [0]:
negative_amounts = spark.sql("""
    SELECT COUNT(*) AS cnt
        FROM dev.claims_project.silver_claims
        WHERE ClaimAmount < 0
""").collect()[0]["cnt"]

assert negative_amounts == 0, "Negative ClaimAmount values found"

In [0]:
duplicate_claims = spark.sql("""
SELECT COUNT(*) AS cnt
    FROM (
        SELECT ClaimID
        FROM dev.claims_project.silver_claims
        GROUP BY ClaimID
        HAVING COUNT(*) > 1
    )""").collect()[0]["cnt"]

assert duplicate_claims == 0, "Duplicate ClaimIDs found"

In [0]:
invalid_scores = spark.sql("""
    SELECT COUNT(*) AS cnt
    FROM dev.claims_project.gold_suspicious_claims
    WHERE suspicion_score < 0
    OR suspicion_score > 5
""").collect()[0]["cnt"]

assert invalid_scores == 0, "Invalid suspicion scores found"
